# Degron Footprint from Python

plan-system 3.4. Uses the local service through `api.client.Client` (standard library only).

Scores describe one reporter assay (GPS, Zhang et al. 2023) and a **development benchmark, not external validation**; they are not probabilities of disease. Run the cells in order with the API running (`make docker-up`) and `PYTHONPATH=system/app/backend`.

In [ ]:
import os
from api.client import Client

BASE = os.environ.get('DF_API_URL', 'http://127.0.0.1:8080')
c = Client(BASE)
print(c.health()['status'], [m['model_id'] for m in c.models() if m['role'] == 'effect'])

## 1. Score one tile
The domain status and warnings come first; read them before the numbers.

In [ ]:
SEQ = os.environ.get('DF_EXAMPLE_SEQUENCE', 'DIMFLVDSSGSIGPENFSKMKTFMKNLV')
r = c.predict(sequence=SEQ)
print('domain:', r['domain']['status'])
print('warnings:', [w['code'] for w in r['warnings']])
print('evidence:', r['provenance']['evidence_label'])
top = sorted(r['cells'], key=lambda x: x['rank_in_tile'])[:5]
for x in top:
    print(f"{x['wt_aa']}{x['position']}{x['mut_aa']}  raw {x['score_dpsi_raw']:.3f}  P {x['p_dpsi_ge_1_calibrated']:.3f}")

## 2. Save it in a project and submit a small job

In [ ]:
p = c.create_project('notebook example')
a = c.save_analysis(p['project_id'], sequence=SEQ)
job = c.submit_job(p['project_id'], {'mode': 'peptides', 'peptides': [{'input_id': 'ex1', 'sequence': SEQ}]})
print('analysis', a['analysis_id'][:8], 'job', job['job_id'][:8], job['state'])

In [ ]:
job = c.wait_job(job['job_id'], poll=0.5, timeout=float(os.environ.get('DF_NOTEBOOK_WAIT', '600')))
print(job['state'], job['counts'])

## 3. Results with their provenance
Every export carries the model id, weights hash, data sources and the disclaimer.

In [ ]:
csv_text = c.export_analysis(a['analysis_id'], 'csv')
print('\n'.join(l for l in csv_text.splitlines() if l.startswith('# model_id') or l.startswith('# evidence_label')))
print(len([l for l in csv_text.splitlines() if not l.startswith('#')]) - 1, 'substitutions')